# SCD using NORML CODE on DELTA TABLE

## SCD Type 1 – Delta Lake MERGE (Overwrite Strategy)
**What SCD Type 1 Means**
- Maintain only the latest state
- No history
- Old values are overwritten
- One row per business key

**Assumptions**
- Business key: customer_id
- Attributes: name, email
- No start_date, end_date, is_current

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "target_table")

(
    target.alias("t")
    .merge(
        source.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(
        condition="t.name <> s.name OR t.email <> s.email",  ## here it will check that is there any actual changes happened in the source data, This avoid unnecessary updates 
        set={
            "name": "s.name",
            "email": "s.email",
            "updated_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "created_at": "current_timestamp()"
    })
    .execute()
)



## SCD Type 2 – Delta Lake MERGE
### Assumptions

- Business key: customer_id

- Columns tracked: name, email

- SCD columns:
> - is_current (BOOLEAN)
> - start_date (TIMESTAMP)
> - end_date (TIMESTAMP)

### Step1: Prepare source with SCD metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit

source_scd = (
    source
    .withColumn("start_date", current_timestamp())
    .withColumn("end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)


### Step 2: SCD Type 2 MERGE

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "target_table")
# target = DElta.forPath(spark, "/tmp/target_table")  --> s the source is delta file, we can also use this

(
    target.alias("t")
    .merge(
        source_scd.alias("s"),
        """
        t.customer_id = s.customer_id
        AND t.is_current = true
        """
    )
    # 1️⃣ Expire old record
    .whenMatchedUpdate(
        condition="""
        t.name <> s.name OR t.email <> s.email
        """,
        set={
            "is_current": "false",
            "end_date": "current_timestamp()"
        }
    )
    # 2️⃣ Insert new version
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "start_date": "s.start_date",
        "end_date": "s.end_date",
        "is_current": "s.is_current"
    })
    .execute()
)
